In [3]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

print("--- PHASE 1: KAGGLE GRANDMASTER PROBE ---")

# 2. ULTIMATE FEATURE ENGINEERING
def engineer_features(train_df, test_df):
    train_df['is_test'] = 0
    test_df['is_test'] = 1
    df = pd.concat([train_df, test_df], ignore_index=True)

    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    # Missingness
    for col in combine_cols:
        df[f'{col}_ismissing'] = df[col].isnull().astype(int)

    # Advanced Biomechanics & Ratios
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)

    # New: Leverage Ratios
    df['Weight_to_Speed'] = df['Weight'] / df['Sprint_40yd']
    df['Jump_to_Height'] = df['Vertical_Jump'] / df['Height']

    # Positional Relativism
    for col in combine_cols + ['Height', 'Weight', 'BMI']:
        df[f'{col}_Pos_Mean'] = df.groupby('Position')[col].transform('mean')
        df[f'{col}_Pos_Diff'] = df[col] - df[f'{col}_Pos_Mean']

    # Categoricals - Create Native for CatBoost, LabelEncoded for LGBM/XGB
    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[f'{col}_Freq'] = df.groupby(col)['Id'].transform('count')

        # Native strings for CatBoost
        df[f'{col}_Native'] = df[col].fillna('Unknown').astype(str)

        # Integers for LGBM & XGB
        le = LabelEncoder()
        df[f'{col}_LE'] = le.fit_transform(df[f'{col}_Native'])

    # THE FIX: Drop the original raw text columns so they don't crash LightGBM/XGBoost
    df.drop(columns=cat_cols, inplace=True)

    train_out = df[df['is_test'] == 0].drop(['is_test'], axis=1)
    test_out = df[df['is_test'] == 1].drop(['is_test', 'Drafted'], axis=1)
    return train_out, test_out

train_df, test_df = engineer_features(train, test)

# Base Feature Lists
cat_native = [c for c in train_df.columns if '_Native' in c]
cat_le = [c for c in train_df.columns if '_LE' in c]

# Features for CatBoost (Strings)
features_cb = [c for c in train_df.columns if c not in ['Id', 'Drafted'] + cat_le]
X_cb = train_df[features_cb]

# Features for LGBM/XGB (Encoded)
features_tree = [c for c in train_df.columns if c not in ['Id', 'Drafted'] + cat_native]
X_tree = train_df[features_tree]

y = train_df['Drafted']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. ADVERSARIAL VALIDATION PROBE
print("\n[1/4] Running Adversarial Validation (Train vs Test Distribution Check)...")
adv_df = pd.concat([train_df[features_tree], test_df[features_tree]])
adv_y = np.array([0]*len(train_df) + [1]*len(test_df))
adv_model = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
adv_auc = np.mean(cross_val_score(adv_model, adv_df, adv_y, cv=3, scoring='roc_auc'))
print(f"Adversarial AUC: {adv_auc:.4f} (If this is > 0.70, we have data drift. If ~0.50, perfect.)")

# 4. OPTUNA GAUNTLET
optuna.logging.set_verbosity(optuna.logging.WARNING)
n_optuna_trials = 25

print(f"\n[2/4] Tuning LightGBM ({n_optuna_trials} trials)...")
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'verbose': -1, 'random_state': 42
    }
    model = LGBMClassifier(**params)
    return np.mean(cross_val_score(model, X_tree, y, cv=cv, scoring='roc_auc'))

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=n_optuna_trials)
print("Best LGBM Params:", study_lgb.best_params)

print(f"\n[3/4] Tuning XGBoost ({n_optuna_trials} trials)...")
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'eval_metric': 'auc', 'random_state': 42
    }
    model = XGBClassifier(**params)
    return np.mean(cross_val_score(model, X_tree, y, cv=cv, scoring='roc_auc'))

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=n_optuna_trials)
print("Best XGB Params:", study_xgb.best_params)

print(f"\n[4/4] Tuning CatBoost ({n_optuna_trials} trials)...")
def objective_cb(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'eval_metric': 'AUC', 'verbose': 0, 'random_seed': 42
    }
    auc_scores = []
    for tr_idx, va_idx in cv.split(X_cb, y):
        X_tr, y_tr = X_cb.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_cb.iloc[va_idx], y.iloc[va_idx]
        train_pool = Pool(X_tr, y_tr, cat_features=cat_native)
        val_pool = Pool(X_va, y_va, cat_features=cat_native)
        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=30)
        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
    return np.mean(auc_scores)

study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=n_optuna_trials)
print("Best CatBoost Params:", study_cb.best_params)

print("\n--- DONE! PLEASE COPY AND PASTE EVERYTHING ABOVE THIS LINE BACK TO ME ---")

--- PHASE 1: KAGGLE GRANDMASTER PROBE ---

[1/4] Running Adversarial Validation (Train vs Test Distribution Check)...
Adversarial AUC: 0.5221 (If this is > 0.70, we have data drift. If ~0.50, perfect.)

[2/4] Tuning LightGBM (25 trials)...
Best LGBM Params: {'n_estimators': 532, 'learning_rate': 0.01554652701800679, 'max_depth': 3, 'subsample': 0.9314639318038497, 'colsample_bytree': 0.6072948098974517}

[3/4] Tuning XGBoost (25 trials)...
Best XGB Params: {'n_estimators': 701, 'learning_rate': 0.010705226164666546, 'max_depth': 3, 'subsample': 0.7130091257790424, 'colsample_bytree': 0.5098970084354815}

[4/4] Tuning CatBoost (25 trials)...
Best CatBoost Params: {'iterations': 602, 'learning_rate': 0.03494020499597501, 'depth': 7, 'l2_leaf_reg': 8.214135553841935}

--- DONE! PLEASE COPY AND PASTE EVERYTHING ABOVE THIS LINE BACK TO ME ---


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

print("--- PHASE 2: LEVEL-2 STACKING ENSEMBLE ---")

# 2. ULTIMATE FEATURE ENGINEERING (Exact same as Phase 1)
def engineer_features(train_df, test_df):
    train_df['is_test'] = 0
    test_df['is_test'] = 1
    df = pd.concat([train_df, test_df], ignore_index=True)

    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    for col in combine_cols:
        df[f'{col}_ismissing'] = df[col].isnull().astype(int)

    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)
    df['Weight_to_Speed'] = df['Weight'] / df['Sprint_40yd']
    df['Jump_to_Height'] = df['Vertical_Jump'] / df['Height']

    for col in combine_cols + ['Height', 'Weight', 'BMI']:
        df[f'{col}_Pos_Mean'] = df.groupby('Position')[col].transform('mean')
        df[f'{col}_Pos_Diff'] = df[col] - df[f'{col}_Pos_Mean']

    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[f'{col}_Freq'] = df.groupby(col)['Id'].transform('count')
        df[f'{col}_Native'] = df[col].fillna('Unknown').astype(str)
        le = LabelEncoder()
        df[f'{col}_LE'] = le.fit_transform(df[f'{col}_Native'])

    df.drop(columns=cat_cols, inplace=True)

    train_out = df[df['is_test'] == 0].drop(['is_test'], axis=1)
    test_out = df[df['is_test'] == 1].drop(['is_test', 'Drafted'], axis=1)
    return train_out, test_out

train_df, test_df = engineer_features(train, test)

# Base Feature Lists
cat_native = [c for c in train_df.columns if '_Native' in c]
cat_le = [c for c in train_df.columns if '_LE' in c]

# Features for CatBoost (Strings)
features_cb = [c for c in train_df.columns if c not in ['Id', 'Drafted'] + cat_le]
X_cb = train_df[features_cb]
X_test_cb = test_df[features_cb]

# Features for LGBM/XGB (Encoded)
features_tree = [c for c in train_df.columns if c not in ['Id', 'Drafted'] + cat_native]
X_tree = train_df[features_tree]
X_test_tree = test_df[features_tree]

y = train_df['Drafted']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. INITIALIZE OPTIMIZED BASE MODELS
lgb_params = {'n_estimators': 532, 'learning_rate': 0.01554652701800679, 'max_depth': 3, 'subsample': 0.9314639318038497, 'colsample_bytree': 0.6072948098974517, 'random_state': 42, 'verbose': -1}
xgb_params = {'n_estimators': 701, 'learning_rate': 0.010705226164666546, 'max_depth': 3, 'subsample': 0.7130091257790424, 'colsample_bytree': 0.5098970084354815, 'random_state': 42, 'eval_metric': 'auc'}
cb_params = {'iterations': 602, 'learning_rate': 0.03494020499597501, 'depth': 7, 'l2_leaf_reg': 8.214135553841935, 'random_seed': 42, 'eval_metric': 'AUC', 'verbose': 0}

# 4. GENERATE OUT-OF-FOLD (OOF) PREDICTIONS FOR LAYER 1
print("\n[1/3] Training Layer-1 Base Models and generating OOF Meta-Features...")
oof_lgb = np.zeros(len(train_df))
oof_xgb = np.zeros(len(train_df))
oof_cb = np.zeros(len(train_df))

test_lgb = np.zeros(len(test_df))
test_xgb = np.zeros(len(test_df))
test_cb = np.zeros(len(test_df))

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_tree, y)):
    print(f"  -> Processing Fold {fold+1}/5")
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    # LGBM
    m_lgb = LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tree.iloc[tr_idx], y_tr)
    oof_lgb[va_idx] = m_lgb.predict_proba(X_tree.iloc[va_idx])[:, 1]
    test_lgb += m_lgb.predict_proba(X_test_tree)[:, 1] / 5.0

    # XGB
    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tree.iloc[tr_idx], y_tr)
    oof_xgb[va_idx] = m_xgb.predict_proba(X_tree.iloc[va_idx])[:, 1]
    test_xgb += m_xgb.predict_proba(X_test_tree)[:, 1] / 5.0

    # CatBoost
    train_pool = Pool(X_cb.iloc[tr_idx], y_tr, cat_features=cat_native)
    val_pool = Pool(X_cb.iloc[va_idx], y_va, cat_features=cat_native)
    test_pool = Pool(X_test_cb, cat_features=cat_native)
    m_cb = CatBoostClassifier(**cb_params)
    m_cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
    oof_cb[va_idx] = m_cb.predict_proba(val_pool)[:, 1]
    test_cb += m_cb.predict_proba(test_pool)[:, 1] / 5.0

print("\nLayer-1 Base Model Individual ROC-AUC Scores:")
print(f"  LightGBM: {roc_auc_score(y, oof_lgb):.5f}")
print(f"  XGBoost:  {roc_auc_score(y, oof_xgb):.5f}")
print(f"  CatBoost: {roc_auc_score(y, oof_cb):.5f}")

# 5. LAYER-2 META-MODEL (LOGISTIC REGRESSION)
print("\n[2/3] Training Layer-2 Meta-Model (Logistic Regression)...")
# Combine base model predictions into new feature matrices
meta_X = np.column_stack((oof_lgb, oof_xgb, oof_cb))
meta_X_test = np.column_stack((test_lgb, test_xgb, test_cb))

meta_oof = np.zeros(len(train_df))
final_test_preds = np.zeros(len(test_df))

# Evaluate Meta-Model using the same Cross-Validation strategy to prevent leakage
for tr_idx, va_idx in cv.split(meta_X, y):
    meta_model = LogisticRegression(C=1.0, random_state=42)
    meta_model.fit(meta_X[tr_idx], y.iloc[tr_idx])

    meta_oof[va_idx] = meta_model.predict_proba(meta_X[va_idx])[:, 1]
    final_test_preds += meta_model.predict_proba(meta_X_test)[:, 1] / 5.0

print(f"\n[!] ULTIMATE LAYER-2 ENSEMBLE ROC-AUC: {roc_auc_score(y, meta_oof):.5f}")

# 6. SUBMISSION
print("\n[3/3] Saving final predictions...")
submission = test[['Id']].copy()
submission['Drafted'] = final_test_preds
submission.to_csv('Data/Submission NFL 2 Optimized.csv', index=False)
print("SUCCESS: File saved to 'Data/submission_grandmaster_stacked.csv'")

--- PHASE 2: LEVEL-2 STACKING ENSEMBLE ---

[1/3] Training Layer-1 Base Models and generating OOF Meta-Features...
  -> Processing Fold 1/5
  -> Processing Fold 2/5
  -> Processing Fold 3/5
  -> Processing Fold 4/5
  -> Processing Fold 5/5

Layer-1 Base Model Individual ROC-AUC Scores:
  LightGBM: 0.83249
  XGBoost:  0.83671
  CatBoost: 0.83569

[2/3] Training Layer-2 Meta-Model (Logistic Regression)...

[!] ULTIMATE LAYER-2 ENSEMBLE ROC-AUC: 0.83736

[3/3] Saving final predictions...
SUCCESS: File saved to 'Data/submission_grandmaster_stacked.csv'


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import category_encoders as ce
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

print("--- PHASE 2 (STRICT ML OPO): LEVEL-2 STACKING ENSEMBLE ---")

# 2. ROW-WISE FEATURE ENGINEERING (Safe from Leakage)
# We only calculate features here that strictly depend on the current row's data.
def engineer_row_features(df):
    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    # Missingness
    for col in combine_cols:
        df[f'{col}_ismissing'] = df[col].isnull().astype(int)

    # Advanced Biomechanics & Ratios
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)
    df['Weight_to_Speed'] = df['Weight'] / df['Sprint_40yd']
    df['Jump_to_Height'] = df['Vertical_Jump'] / df['Height']

    # Cast categoricals to strings to prevent type errors
    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype(str)

    return df

train_df = engineer_row_features(train)
test_df = engineer_row_features(test)

y = train_df['Drafted']
X = train_df.drop(columns=['Id', 'Drafted'])
X_test_base = test_df.drop(columns=['Id'])

cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']
z_score_cols = combine_cols + ['Height', 'Weight', 'BMI', 'Speed_Score']

# 3. INITIALIZE OPTIMIZED BASE MODELS (Using your Optuna params)
lgb_params = {'n_estimators': 532, 'learning_rate': 0.01554652701800679, 'max_depth': 3, 'subsample': 0.9314639318038497, 'colsample_bytree': 0.6072948098974517, 'random_state': 42, 'verbose': -1}
xgb_params = {'n_estimators': 701, 'learning_rate': 0.010705226164666546, 'max_depth': 3, 'subsample': 0.7130091257790424, 'colsample_bytree': 0.5098970084354815, 'random_state': 42, 'eval_metric': 'auc'}
cb_params = {'iterations': 602, 'learning_rate': 0.03494020499597501, 'depth': 7, 'l2_leaf_reg': 8.214135553841935, 'random_seed': 42, 'eval_metric': 'AUC', 'verbose': 0}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

test_lgb = np.zeros(len(X_test_base))
test_xgb = np.zeros(len(X_test_base))
test_cb = np.zeros(len(X_test_base))

print("\n[1/3] Commencing Strict CV Feature Engineering & Layer-1 Training...")

# 4. STRICT CV LOOP (Leakage-Free Transformations)
for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    print(f"  -> Processing Fold {fold+1}/5")

    # 4A. Isolate Fold Data
    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    X_te = X_test_base.copy()

    # 4B. Fold-Specific Positional Z-Scores
    for col in z_score_cols:
        # Calculate stats ONLY on X_tr
        pos_stats = X_tr.groupby('Position')[col].agg(['mean', 'std']).reset_index()
        pos_stats = pos_stats.rename(columns={'mean': f'{col}_Pos_Mean', 'std': f'{col}_Pos_Std'})

        global_mean = X_tr[col].mean()
        global_std = X_tr[col].std()

        # Merge stats onto Tr, Va, Te
        for df_dict in [X_tr, X_va, X_te]:
            temp = df_dict.merge(pos_stats, on='Position', how='left')

            # Fill unseen positions with global train stats
            temp[f'{col}_Pos_Mean'] = temp[f'{col}_Pos_Mean'].fillna(global_mean)
            temp[f'{col}_Pos_Std'] = temp[f'{col}_Pos_Std'].fillna(global_std)

            # Prevent Division by Zero for single-record positions
            temp[f'{col}_Pos_Std'] = temp[f'{col}_Pos_Std'].replace(0, 1e-9)

            # Calculate Z-Score: (Value - Mean) / Std
            df_dict[f'{col}_Pos_Z'] = (df_dict[col] - temp[f'{col}_Pos_Mean']) / temp[f'{col}_Pos_Std']

    # 4C. Target Encoding for LGBM & XGBoost (with smoothing)
    # TargetEncoder calculates the expected probability of the target for each category
    te = ce.TargetEncoder(cols=cat_cols, smoothing=10.0)

    # Fit strictly on X_tr, y_tr
    te.fit(X_tr[cat_cols], y_tr)

    # Create encoded feature sets for the Tree models
    X_tr_tree = X_tr.drop(columns=cat_cols).copy()
    X_va_tree = X_va.drop(columns=cat_cols).copy()
    X_te_tree = X_te.drop(columns=cat_cols).copy()

    X_tr_tree[[f'{c}_TE' for c in cat_cols]] = te.transform(X_tr[cat_cols])
    X_va_tree[[f'{c}_TE' for c in cat_cols]] = te.transform(X_va[cat_cols])
    X_te_tree[[f'{c}_TE' for c in cat_cols]] = te.transform(X_te[cat_cols])

    # 4D. Model Training & Prediction

    # LightGBM (Uses Target Encoded features)
    m_lgb = LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr_tree, y_tr)
    oof_lgb[va_idx] = m_lgb.predict_proba(X_va_tree)[:, 1]
    test_lgb += m_lgb.predict_proba(X_te_tree)[:, 1] / cv.n_splits

    # XGBoost (Uses Target Encoded features)
    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr_tree, y_tr)
    oof_xgb[va_idx] = m_xgb.predict_proba(X_va_tree)[:, 1]
    test_xgb += m_xgb.predict_proba(X_te_tree)[:, 1] / cv.n_splits

    # CatBoost (Uses Native Strings)
    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    val_pool = Pool(X_va, y.iloc[va_idx], cat_features=cat_cols)
    test_pool = Pool(X_te, cat_features=cat_cols)

    m_cb = CatBoostClassifier(**cb_params)
    m_cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
    oof_cb[va_idx] = m_cb.predict_proba(val_pool)[:, 1]
    test_cb += m_cb.predict_proba(test_pool)[:, 1] / cv.n_splits

print("\nLayer-1 Base Model ROC-AUC Scores (Strict Validation):")
print(f"  LightGBM: {roc_auc_score(y, oof_lgb):.5f}")
print(f"  XGBoost:  {roc_auc_score(y, oof_xgb):.5f}")
print(f"  CatBoost: {roc_auc_score(y, oof_cb):.5f}")

# 5. LAYER-2 META-MODEL (LOGISTIC REGRESSION)
print("\n[2/3] Training Layer-2 Meta-Model (Logistic Regression)...")
meta_X = np.column_stack((oof_lgb, oof_xgb, oof_cb))
meta_X_test = np.column_stack((test_lgb, test_xgb, test_cb))

meta_oof = np.zeros(len(X))
final_test_preds = np.zeros(len(X_test_base))

for tr_idx, va_idx in cv.split(meta_X, y):
    meta_model = LogisticRegression(C=1.0, random_state=42)
    meta_model.fit(meta_X[tr_idx], y.iloc[tr_idx])

    meta_oof[va_idx] = meta_model.predict_proba(meta_X[va_idx])[:, 1]
    final_test_preds += meta_model.predict_proba(meta_X_test)[:, 1] / cv.n_splits

print(f"\n[!] ULTIMATE LAYER-2 ENSEMBLE ROC-AUC: {roc_auc_score(y, meta_oof):.5f}")

# 6. SUBMISSION
print("\n[3/3] Saving final predictions...")
submission = test[['Id']].copy()
submission['Drafted'] = final_test_preds
submission.to_csv('Data/Submission 2 Optimization 2.csv', index=False)
print("SUCCESS: File saved to 'Data/submission_strict_grandmaster.csv'")

--- PHASE 2 (STRICT ML OPO): LEVEL-2 STACKING ENSEMBLE ---

[1/3] Commencing Strict CV Feature Engineering & Layer-1 Training...
  -> Processing Fold 1/5
  -> Processing Fold 2/5
  -> Processing Fold 3/5
  -> Processing Fold 4/5
  -> Processing Fold 5/5

Layer-1 Base Model ROC-AUC Scores (Strict Validation):
  LightGBM: 0.82432
  XGBoost:  0.82955
  CatBoost: 0.81647

[2/3] Training Layer-2 Meta-Model (Logistic Regression)...

[!] ULTIMATE LAYER-2 ENSEMBLE ROC-AUC: 0.82217

[3/3] Saving final predictions...
SUCCESS: File saved to 'Data/submission_strict_grandmaster.csv'


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import category_encoders as ce
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

print("--- PHASE 2 (STRICT ML OPO): LEVEL-2 STACKING ENSEMBLE ---")

# 2. ROW-WISE FEATURE ENGINEERING (Safe from Leakage)
def engineer_row_features(df):
    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    # Missingness
    for col in combine_cols:
        df[f'{col}_ismissing'] = df[col].isnull().astype(int)

    # Advanced Biomechanics & Ratios
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)
    df['Weight_to_Speed'] = df['Weight'] / df['Sprint_40yd']
    df['Jump_to_Height'] = df['Vertical_Jump'] / df['Height']

    # Cast categoricals to strings to prevent type errors
    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype(str)

    return df

train_df = engineer_row_features(train)
test_df = engineer_row_features(test)

y = train_df['Drafted']
X = train_df.drop(columns=['Id', 'Drafted'])
X_test_base = test_df.drop(columns=['Id'])

cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']
z_score_cols = combine_cols + ['Height', 'Weight', 'BMI', 'Speed_Score']

# 3. INITIALIZE OPTIMIZED BASE MODELS
lgb_params = {'n_estimators': 532, 'learning_rate': 0.01554652701800679, 'max_depth': 3, 'subsample': 0.9314639318038497, 'colsample_bytree': 0.6072948098974517, 'random_state': 42, 'verbose': -1}
xgb_params = {'n_estimators': 701, 'learning_rate': 0.010705226164666546, 'max_depth': 3, 'subsample': 0.7130091257790424, 'colsample_bytree': 0.5098970084354815, 'random_state': 42, 'eval_metric': 'auc'}
cb_params = {'iterations': 602, 'learning_rate': 0.03494020499597501, 'depth': 7, 'l2_leaf_reg': 8.214135553841935, 'random_seed': 42, 'eval_metric': 'AUC', 'verbose': 0}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

test_lgb = np.zeros(len(X_test_base))
test_xgb = np.zeros(len(X_test_base))
test_cb = np.zeros(len(X_test_base))

print("\n[1/3] Commencing Strict CV Feature Engineering & Layer-1 Training...")

# 4. STRICT CV LOOP (Leakage-Free Transformations)
for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    print(f"  -> Processing Fold {fold+1}/5")

    # 4A. Isolate Fold Data
    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    X_te = X_test_base.copy()

    # 4B. Fold-Specific Positional Z-Scores (SAFE INDEXING)
    for col in z_score_cols:
        # Calculate stats ONLY on X_tr as Pandas Series
        pos_mean = X_tr.groupby('Position')[col].mean()
        pos_std = X_tr.groupby('Position')[col].std().replace(0, 1e-9).fillna(1e-9)

        global_mean = X_tr[col].mean()
        global_std = X_tr[col].std() if X_tr[col].std() != 0 else 1e-9

        # Safely map stats onto Tr, Va, Te without breaking the index
        for df in [X_tr, X_va, X_te]:
            # Map the positional mean/std. If position is unseen, fill with global mean/std
            mapped_mean = df['Position'].map(pos_mean).fillna(global_mean)
            mapped_std = df['Position'].map(pos_std).fillna(global_std)

            # Calculate Z-Score safely
            df[f'{col}_Pos_Z'] = (df[col] - mapped_mean) / mapped_std

    # 4C. Target Encoding for LGBM & XGBoost (with smoothing)
    te = ce.TargetEncoder(cols=cat_cols, smoothing=10.0)

    # Fit strictly on X_tr, y_tr
    te.fit(X_tr[cat_cols], y_tr)

    # Create encoded feature sets for the Tree models
    X_tr_tree = X_tr.drop(columns=cat_cols).copy()
    X_va_tree = X_va.drop(columns=cat_cols).copy()
    X_te_tree = X_te.drop(columns=cat_cols).copy()

    X_tr_tree[[f'{c}_TE' for c in cat_cols]] = te.transform(X_tr[cat_cols])
    X_va_tree[[f'{c}_TE' for c in cat_cols]] = te.transform(X_va[cat_cols])
    X_te_tree[[f'{c}_TE' for c in cat_cols]] = te.transform(X_te[cat_cols])

    # 4D. Model Training & Prediction

    # LightGBM
    m_lgb = LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr_tree, y_tr)
    oof_lgb[va_idx] = m_lgb.predict_proba(X_va_tree)[:, 1]
    test_lgb += m_lgb.predict_proba(X_te_tree)[:, 1] / cv.n_splits

    # XGBoost
    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr_tree, y_tr)
    oof_xgb[va_idx] = m_xgb.predict_proba(X_va_tree)[:, 1]
    test_xgb += m_xgb.predict_proba(X_te_tree)[:, 1] / cv.n_splits

    # CatBoost (Uses Native Strings)
    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    val_pool = Pool(X_va, y.iloc[va_idx], cat_features=cat_cols)
    test_pool = Pool(X_te, cat_features=cat_cols)

    m_cb = CatBoostClassifier(**cb_params)
    m_cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
    oof_cb[va_idx] = m_cb.predict_proba(val_pool)[:, 1]
    test_cb += m_cb.predict_proba(test_pool)[:, 1] / cv.n_splits

print("\nLayer-1 Base Model ROC-AUC Scores (Strict Validation):")
print(f"  LightGBM: {roc_auc_score(y, oof_lgb):.5f}")
print(f"  XGBoost:  {roc_auc_score(y, oof_xgb):.5f}")
print(f"  CatBoost: {roc_auc_score(y, oof_cb):.5f}")

# 5. LAYER-2 META-MODEL (LOGISTIC REGRESSION)
print("\n[2/3] Training Layer-2 Meta-Model (Logistic Regression)...")
meta_X = np.column_stack((oof_lgb, oof_xgb, oof_cb))
meta_X_test = np.column_stack((test_lgb, test_xgb, test_cb))

meta_oof = np.zeros(len(X))
final_test_preds = np.zeros(len(X_test_base))

for tr_idx, va_idx in cv.split(meta_X, y):
    meta_model = LogisticRegression(C=1.0, random_state=42)
    meta_model.fit(meta_X[tr_idx], y.iloc[tr_idx])

    meta_oof[va_idx] = meta_model.predict_proba(meta_X[va_idx])[:, 1]
    final_test_preds += meta_model.predict_proba(meta_X_test)[:, 1] / cv.n_splits

print(f"\n[!] ULTIMATE LAYER-2 ENSEMBLE ROC-AUC: {roc_auc_score(y, meta_oof):.5f}")

# 6. SUBMISSION
print("\n[3/3] Saving final predictions...")
submission = test[['Id']].copy()
submission['Drafted'] = final_test_preds
submission.to_csv('Data/Submission 2 Optimization 3.csv', index=False)
print("SUCCESS: File saved to 'Data/submission_strict_grandmaster.csv'")

In [7]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, percentileofscore
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import category_encoders as ce
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

print("--- PHASE 5: ELITE 10-FOLD OOF PIPELINE ---")

# 2. BASE ROW-WISE FEATURE ENGINEERING (No Leakage)
def init_features(df):
    df['Weight_lbs'] = df['Weight'] * 2.20462
    df['Height_inches'] = df['Height'] * 39.3701

    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    # Isolate missingness signal
    missing_flags = []
    for col in combine_cols:
        flag_col = f'{col}_ismissing'
        df[flag_col] = df[col].isnull().astype(int)
        missing_flags.append(flag_col)

    df['Combine_Missing_Count'] = df[missing_flags].sum(axis=1)

    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype(str)

    return df

train_df = init_features(train)
test_df = init_features(test)

y = train_df['Drafted']
X = train_df.drop(columns=['Id', 'Drafted'])
X_test_base = test_df.drop(columns=['Id'])

combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']
z_score_cols = combine_cols + ['Height_inches', 'Weight_lbs']

low_card_cats = ['Player_Type', 'Position_Type', 'Position']
high_card_cats = ['School']

# 3. INITIALIZE MODELS
# Note: Using robust baseline parameters, letting the ensemble weights do the heavy lifting
lgb_params = {'n_estimators': 1200, 'learning_rate': 0.015, 'max_depth': 4, 'subsample': 0.85, 'colsample_bytree': 0.65, 'random_state': 42, 'n_jobs': -1}
xgb_params = {'n_estimators': 1200, 'learning_rate': 0.01, 'max_depth': 4, 'subsample': 0.8, 'colsample_bytree': 0.6, 'random_state': 42, 'eval_metric': 'auc', 'enable_categorical': True, 'n_jobs': -1}
cb_params = {'iterations': 1200, 'learning_rate': 0.02, 'depth': 6, 'l2_leaf_reg': 5.0, 'random_seed': 42, 'eval_metric': 'AUC', 'verbose': 0}

# 10-Fold Stratified CV
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

test_lgb = np.zeros(len(X_test_base))
test_xgb = np.zeros(len(X_test_base))
test_cb = np.zeros(len(X_test_base))

print("\n[1/3] Executing 10-Fold CV (Imputation, eCDF, LOOE)...")

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    print(f"  -> Processing Fold {fold+1}/10")

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    X_te = X_test_base.copy()

    # A. Cluster-Based Imputation (Strictly on X_tr Medians)
    for col in combine_cols:
        pos_medians = X_tr.groupby('Position')[col].median()
        global_median = X_tr[col].median()

        for df in [X_tr, X_va, X_te]:
            # Fill NaNs with Positional Median, fallback to Global Median
            mapped_medians = df['Position'].map(pos_medians).fillna(global_median)
            df[col] = df[col].fillna(mapped_medians)

    # B. Advanced Biomechanical Ratios (Post-Imputation)
    for df in [X_tr, X_va, X_te]:
        df['BMI'] = (df['Weight_lbs'] / (df['Height_inches'] ** 2)) * 703
        df['Burst_Score'] = df['Vertical_Jump'] + df['Broad_Jump']
        df['Speed_Score'] = (df['Weight_lbs'] * 200) / (df['Sprint_40yd'] ** 4)
        df['Power_Success'] = df['Bench_Press_Reps'] * df['Weight_lbs']

    z_score_cols_extended = z_score_cols + ['BMI', 'Burst_Score', 'Speed_Score', 'Power_Success']

    # C. Robust Positional Z-Scores & eCDF
    for col in z_score_cols_extended:
        # Clip tails on Train only to get robust Mean/Std
        clipped_tr = X_tr.copy()
        for pos in clipped_tr['Position'].unique():
            mask = clipped_tr['Position'] == pos
            p01 = clipped_tr.loc[mask, col].quantile(0.01)
            p99 = clipped_tr.loc[mask, col].quantile(0.99)
            clipped_tr.loc[mask, col] = clipped_tr.loc[mask, col].clip(p01, p99)

        pos_mean = clipped_tr.groupby('Position')[col].mean()
        pos_std = clipped_tr.groupby('Position')[col].std().replace(0, 1e-9).fillna(1e-9)

        for df in [X_tr, X_va, X_te]:
            mapped_mean = df['Position'].map(pos_mean).fillna(clipped_tr[col].mean())
            mapped_std = df['Position'].map(pos_std).fillna(clipped_tr[col].std() + 1e-9)
            df[f'{col}_Pos_Z'] = (df[col] - mapped_mean) / mapped_std

            # eCDF Proxy (Using Z-Score CDF approximation for speed, as computing true eCDF row-by-row is slow)
            # A standard normal CDF approximates the percentile perfectly for Z-scores
            from scipy.stats import norm
            df[f'{col}_eCDF'] = norm.cdf(df[f'{col}_Pos_Z'])

    # D. Leave-One-Out Encoding (LOOE) for School
    looe = ce.LeaveOneOutEncoder(cols=high_card_cats, sigma=0.05, random_state=42)
    X_tr_encoded = looe.fit_transform(X_tr, y_tr)
    X_va_encoded = looe.transform(X_va)
    X_te_encoded = looe.transform(X_te)

    # Prepare Tree features
    for col in low_card_cats:
        X_tr_encoded[col] = X_tr_encoded[col].astype('category')
        X_va_encoded[col] = pd.Categorical(X_va_encoded[col], categories=X_tr_encoded[col].cat.categories)
        X_te_encoded[col] = pd.Categorical(X_te_encoded[col], categories=X_tr_encoded[col].cat.categories)

    # E. Model Training
    # LightGBM
    m_lgb = LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr_encoded, y_tr)
    oof_lgb[va_idx] = m_lgb.predict_proba(X_va_encoded)[:, 1]
    test_lgb += m_lgb.predict_proba(X_te_encoded)[:, 1] / cv.n_splits

    # XGBoost
    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr_encoded, y_tr)
    oof_xgb[va_idx] = m_xgb.predict_proba(X_va_encoded)[:, 1]
    test_xgb += m_xgb.predict_proba(X_te_encoded)[:, 1] / cv.n_splits

    # CatBoost (Uses Raw Categoricals - Drops the LOOE columns, uses original X_tr)
    # CatBoost's internal Ordered Target Encoding is state-of-the-art, feed it raw strings.
    cb_features = [c for c in X_tr.columns]
    train_pool = Pool(X_tr[cb_features], y_tr, cat_features=low_card_cats + high_card_cats)
    val_pool = Pool(X_va[cb_features], y.iloc[va_idx], cat_features=low_card_cats + high_card_cats)
    test_pool = Pool(X_te[cb_features], cat_features=low_card_cats + high_card_cats)

    m_cb = CatBoostClassifier(**cb_params)
    m_cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)
    oof_cb[va_idx] = m_cb.predict_proba(val_pool)[:, 1]
    test_cb += m_cb.predict_proba(test_pool)[:, 1] / cv.n_splits

print("\nLayer-1 Base Model ROC-AUC Scores:")
print(f"  LightGBM: {roc_auc_score(y, oof_lgb):.5f}")
print(f"  XGBoost:  {roc_auc_score(y, oof_xgb):.5f}")
print(f"  CatBoost: {roc_auc_score(y, oof_cb):.5f}")

# 4. OPTUNA LAYER-2 META-LEARNER
print("\n[2/3] Executing Optuna Rank-Blending Optimization...")
optuna.logging.set_verbosity(optuna.logging.WARNING)

rank_oof_lgb = rankdata(oof_lgb) / len(oof_lgb)
rank_oof_xgb = rankdata(oof_xgb) / len(oof_xgb)
rank_oof_cb  = rankdata(oof_cb) / len(oof_cb)

def objective(trial):
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_cb = trial.suggest_float('w_cb', 0.0, 1.0)

    # Normalize weights
    total = w_lgb + w_xgb + w_cb
    if total == 0: return 0.5
    w_lgb, w_xgb, w_cb = w_lgb/total, w_xgb/total, w_cb/total

    blend = (w_lgb * rank_oof_lgb) + (w_xgb * rank_oof_xgb) + (w_cb * rank_oof_cb)
    return roc_auc_score(y, blend)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=300)

best_w = study.best_params
total_w = sum(best_w.values())
w_lgb = best_w['w_lgb'] / total_w
w_xgb = best_w['w_xgb'] / total_w
w_cb  = best_w['w_cb'] / total_w

print(f"\n[!] Optimal Optuna Weights Found:")
print(f"    LightGBM Weight: {w_lgb:.4f}")
print(f"    XGBoost Weight:  {w_xgb:.4f}")
print(f"    CatBoost Weight: {w_cb:.4f}")

# 5. FINAL INFERENCE & SUBMISSION
final_oof = (w_lgb * rank_oof_lgb) + (w_xgb * rank_oof_xgb) + (w_cb * rank_oof_cb)
print(f"\n[!] ULTIMATE 10-FOLD OPTIMIZED ROC-AUC: {roc_auc_score(y, final_oof):.5f}")

rank_test_lgb = rankdata(test_lgb) / len(test_lgb)
rank_test_xgb = rankdata(test_xgb) / len(test_xgb)
rank_test_cb  = rankdata(test_cb) / len(test_cb)

final_test_preds = (w_lgb * rank_test_lgb) + (w_xgb * rank_test_xgb) + (w_cb * rank_test_cb)

print("\n[3/3] Saving final tournament predictions...")
submission = test[['Id']].copy()
submission['Drafted'] = final_test_preds
submission.to_csv('Data/Submission 2 Optimization 4.csv', index=False)
print("SUCCESS: File saved to 'Data/Submission 2 Optimization 4.csv")

--- PHASE 5: ELITE 10-FOLD OOF PIPELINE ---

[1/3] Executing 10-Fold CV (Imputation, eCDF, LOOE)...
  -> Processing Fold 1/10
  -> Processing Fold 2/10
  -> Processing Fold 3/10
  -> Processing Fold 4/10
  -> Processing Fold 5/10
  -> Processing Fold 6/10
  -> Processing Fold 7/10
  -> Processing Fold 8/10
  -> Processing Fold 9/10
  -> Processing Fold 10/10

Layer-1 Base Model ROC-AUC Scores:
  LightGBM: 0.82610
  XGBoost:  0.83753
  CatBoost: 0.81405

[2/3] Executing Optuna Rank-Blending Optimization...

[!] Optimal Optuna Weights Found:
    LightGBM Weight: 0.0001
    XGBoost Weight:  0.7279
    CatBoost Weight: 0.2720

[!] ULTIMATE 10-FOLD OPTIMIZED ROC-AUC: 0.83957

[3/3] Saving final tournament predictions...
SUCCESS: File saved to 'Data/Submission 2 Optimization 4.csv


In [2]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import category_encoders as ce
import warnings
warnings.filterwarnings('ignore')

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. LOAD DATA
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

print("--- FINAL PHASE: HIGH-CEILING STACK + CATBOOST TOURNAMENT + LEAK-SAFE PSEUDO ---")

def engineer_global_features(df, train_df_for_counts=None):
    df = df.copy()
    df['Weight_lbs'] = df['Weight'] * 2.20462
    df['Height_inches'] = df['Height'] * 39.3701

    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    # Missingness Profile
    df['Combine_Missing_Count'] = df[combine_cols].isnull().sum(axis=1)
    df['Did_Any_Agility'] = ((df['Agility_3cone'].notnull()) | (df['Shuttle'].notnull())).astype(int)

    # Safe Math for Physics
    safe_40yd = df['Sprint_40yd'].replace(0, np.nan).fillna(999)
    safe_height = df['Height_inches'].replace(0, np.nan).fillna(999)
    safe_weight = df['Weight_lbs'].replace(0, np.nan).fillna(1)

    # Advanced Physics / Scouting Composites
    df['BMI'] = (df['Weight_lbs'] / (safe_height ** 2)) * 703
    df['Speed_Score'] = np.where(df['Sprint_40yd'] > 0, (df['Weight_lbs'] * 200) / (safe_40yd ** 4), 0)
    df['Burst_Score'] = np.where((df['Vertical_Jump'] > 0) & (df['Broad_Jump'] > 0), df['Vertical_Jump'] + df['Broad_Jump'], 0)
    df['Power_Success'] = np.where(df['Bench_Press_Reps'] > 0, df['Bench_Press_Reps'] * df['Weight_lbs'], 0)

    # Momentum (Mass x Velocity Proxy)
    df['Momentum'] = df['Weight_lbs'] / safe_40yd
    # Explode proxy (How far can you jump per lb of bodyweight)
    df['Broad_per_lb'] = df['Broad_Jump'].fillna(0) / safe_weight

    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype(str)

    return df

# --- IMPORTANT EXECUTIONS OUTSIDE THE FUNCTION ---
train_df = engineer_global_features(train)
test_df = engineer_global_features(test)

# THE SCHOOL TIER HACK (Fit strictly on Train, mapped to both)
school_counts = train_df['School'].value_counts()
train_df['School_Volume'] = train_df['School'].map(school_counts)
test_df['School_Volume'] = test_df['School'].map(school_counts).fillna(1) # Unseen schools get 1

# Interaction: Missingness x School Prestige (The "Skipped Drill Privilege")
train_df['Missing_x_Volume'] = train_df['Combine_Missing_Count'] * np.log1p(train_df['School_Volume'])
test_df['Missing_x_Volume'] = test_df['Combine_Missing_Count'] * np.log1p(test_df['School_Volume'])

y = train_df['Drafted'].reset_index(drop=True)
X = train_df.drop(columns=['Id', 'Drafted', 'Weight', 'Height']).reset_index(drop=True)
X_test_base = test_df.drop(columns=['Id', 'Weight', 'Height']).reset_index(drop=True)

train_pos = int((y == 1).sum())
train_neg = int((y == 0).sum())
print(f"Local target split: {train_pos} class-1 / {train_neg} class-0 ({y.mean():.2%} class-1).")
print("Note: pseudo-label bottleneck uses this local empirical split, not the stale 20/80 assumption.")

combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
low_card_cats = ['Player_Type', 'Position_Type', 'Position']
high_card_cats = ['School']

# 3. IN-FOLD PROCESSING ENGINE
def process_fold(X_tr, X_va, X_te, y_tr):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()

    # Strict Leak-Free Positional Deltas
    for col in combine_cols:
        pos_means = X_tr.groupby('Position')[col].mean()
        global_mean = X_tr[col].mean()

        for df in [X_tr, X_va, X_te]:
            mapped_means = df['Position'].map(pos_means).fillna(global_mean)
            df[f'{col}_vs_Pos_Avg'] = df[col] - mapped_means
            df[f'{col}_vs_Pos_Avg'] = df[f'{col}_vs_Pos_Avg'].fillna(0)
            df[col] = df[col].fillna(-999)

    # Target Encoding
    looe = ce.LeaveOneOutEncoder(cols=high_card_cats, sigma=0.0)
    X_tr[high_card_cats] = looe.fit_transform(X_tr[high_card_cats], y_tr)
    X_va[high_card_cats] = looe.transform(X_va[high_card_cats])
    X_te[high_card_cats] = looe.transform(X_te[high_card_cats])

    # THE FIX: Unified Native Categories to prevent NaN on rare out-of-fold values
    for col in low_card_cats:
        unified_categories = pd.unique(pd.concat([X_tr[col], X_va[col], X_te[col]]))
        X_tr[col] = pd.Categorical(X_tr[col], categories=unified_categories)
        X_va[col] = pd.Categorical(X_va[col], categories=unified_categories)
        X_te[col] = pd.Categorical(X_te[col], categories=unified_categories)

    return X_tr, X_va, X_te


def build_ultimate_catboost_matrix():
    target = train['Drafted'].reset_index(drop=True)
    train_ids = train['Id']
    test_ids = test['Id']
    df = pd.concat([train.drop(columns=['Drafted']), test], axis=0).reset_index(drop=True)

    physical_metrics = ['Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
                        'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Age']

    for col in physical_metrics:
        if df[col].isnull().sum() > 0:
            df[f'{col}_Is_Missing'] = df[col].isnull().astype(int)

    for col in physical_metrics:
        df[col] = df[col].fillna(df.groupby('Position')[col].transform('median'))
        df[col] = df[col].fillna(df[col].median())

    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Speed_Weight_Ratio'] = df['Weight'] / df['Sprint_40yd']
    df['Explosiveness_Index'] = df['Vertical_Jump'] * df['Broad_Jump']
    df['Power_Score'] = df['Bench_Press_Reps'] * df['Weight']
    df['School_Combine_Count'] = df.groupby('School')['Id'].transform('count')
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)

    for col in physical_metrics:
        df[f'{col}_Pos_Mean'] = df.groupby('Position')[col].transform('mean')
        df[f'{col}_Pos_Diff'] = df[col] - df[f'{col}_Pos_Mean']
        df.drop(columns=[f'{col}_Pos_Mean'], inplace=True)

    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype('category')

    X_train_ult = df[df['Id'].isin(train_ids)].drop(columns=['Id']).reset_index(drop=True)
    X_test_ult = df[df['Id'].isin(test_ids)].drop(columns=['Id']).reset_index(drop=True)
    return X_train_ult, X_test_ult, target, cat_cols


# 4. OPTUNA: THE 3-WAY SPLIT
print("\n[1/6] Running 3-Way Split Optuna Tuning...")

def tune_models():
    cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    def objective(trial, model_type):
        scores = []
        for tr_idx, va_idx in cv_tune.split(X, y):
            # Main Split
            X_main, y_main = X.iloc[tr_idx], y.iloc[tr_idx]
            X_va_raw, y_va_t = X.iloc[va_idx], y.iloc[va_idx]

            # The 3-Way Split
            tr_sub_idx, es_sub_idx = train_test_split(
                np.arange(len(X_main)), test_size=0.15, stratify=y_main, random_state=42
            )
            X_tr_raw, y_tr_t = X_main.iloc[tr_sub_idx], y_main.iloc[tr_sub_idx]
            X_es_raw, y_es_t = X_main.iloc[es_sub_idx], y_main.iloc[es_sub_idx]

            # Leak-Free Processing
            X_tr_t, X_es_t, X_va_t = process_fold(X_tr_raw, X_es_raw, X_va_raw, y_tr_t)

            if model_type == 'lgb':
                params = {'n_estimators': 1000, 'learning_rate': trial.suggest_float('lr', 0.01, 0.05, log=True), 'max_depth': trial.suggest_int('depth', 3, 5), 'subsample': trial.suggest_float('sub', 0.6, 0.9), 'colsample_bytree': trial.suggest_float('col', 0.5, 0.9), 'min_child_weight': trial.suggest_int('mcw', 5, 15), 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
                m = LGBMClassifier(**params)
                m.fit(X_tr_t, y_tr_t, eval_set=[(X_es_t, y_es_t)], callbacks=[early_stopping(50, verbose=False), log_evaluation(0)])
                scores.append(roc_auc_score(y_va_t, m.predict_proba(X_va_t)[:, 1]))

            elif model_type == 'xgb':
                params = {'n_estimators': 1000, 'learning_rate': trial.suggest_float('lr', 0.01, 0.05, log=True), 'max_depth': trial.suggest_int('depth', 3, 5), 'subsample': trial.suggest_float('sub', 0.6, 0.9), 'colsample_bytree': trial.suggest_float('col', 0.5, 0.9), 'min_child_weight': trial.suggest_int('mcw', 5, 15), 'random_state': 42, 'eval_metric': 'auc', 'enable_categorical': True, 'early_stopping_rounds': 50, 'n_jobs': -1}
                m = XGBClassifier(**params)
                m.fit(X_tr_t, y_tr_t, eval_set=[(X_es_t, y_es_t)], verbose=False)
                scores.append(roc_auc_score(y_va_t, m.predict_proba(X_va_t)[:, 1]))

            elif model_type == 'cb':
                params = {'iterations': 1000, 'learning_rate': trial.suggest_float('lr', 0.02, 0.08, log=True), 'depth': trial.suggest_int('depth', 4, 6), 'l2_leaf_reg': trial.suggest_float('l2', 1.0, 8.0), 'min_data_in_leaf': trial.suggest_int('mdl', 5, 15), 'random_seed': 42, 'eval_metric': 'AUC', 'early_stopping_rounds': 50, 'verbose': 0}
                cat_idx = [X_tr_t.columns.get_loc(c) for c in low_card_cats]
                train_pool = Pool(X_tr_t, y_tr_t, cat_features=cat_idx)
                es_pool = Pool(X_es_t, y_es_t, cat_features=cat_idx)
                va_pool = Pool(X_va_t, cat_features=cat_idx)

                m = CatBoostClassifier(**params)
                m.fit(train_pool, eval_set=es_pool)
                scores.append(roc_auc_score(y_va_t, m.predict_proba(va_pool)[:, 1]))

        return np.mean(scores)

    study_lgb = optuna.create_study(direction='maximize')
    study_lgb.optimize(lambda t: objective(t, 'lgb'), n_trials=15)

    study_xgb = optuna.create_study(direction='maximize')
    study_xgb.optimize(lambda t: objective(t, 'xgb'), n_trials=15)

    study_cb = optuna.create_study(direction='maximize')
    study_cb.optimize(lambda t: objective(t, 'cb'), n_trials=15)

    return study_lgb.best_params, study_xgb.best_params, study_cb.best_params

p_lgb, p_xgb, p_cb = tune_models()

lgb_params = {'n_estimators': 2000, 'learning_rate': p_lgb['lr'], 'max_depth': p_lgb['depth'], 'subsample': p_lgb['sub'], 'colsample_bytree': p_lgb['col'], 'min_child_weight': p_lgb['mcw'], 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
xgb_params = {'n_estimators': 2000, 'learning_rate': p_xgb['lr'], 'max_depth': p_xgb['depth'], 'subsample': p_xgb['sub'], 'colsample_bytree': p_xgb['col'], 'min_child_weight': p_xgb['mcw'], 'random_state': 42, 'eval_metric': 'auc', 'enable_categorical': True, 'early_stopping_rounds': 100, 'n_jobs': -1}
cb_params = {'iterations': 2000, 'learning_rate': p_cb['lr'], 'depth': p_cb['depth'], 'l2_leaf_reg': p_cb['l2'], 'min_data_in_leaf': p_cb['mdl'], 'random_seed': 42, 'eval_metric': 'AUC', 'early_stopping_rounds': 100, 'verbose': 0}

ultimate_cb_params = {
    'iterations': 1183,
    'learning_rate': 0.010492928203031996,
    'depth': 4,
    'l2_leaf_reg': 4.5594175959311425,
    'min_data_in_leaf': 7,
    'random_seed': 42,
    'verbose': 0
}


def run_base_cv(pseudo_X=None, pseudo_y=None, stage_name='Clean CV'):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    n_splits = cv.get_n_splits()
    has_pseudo = pseudo_X is not None and pseudo_y is not None and len(pseudo_X) > 0

    oof_lgb, test_lgb = np.zeros(len(X)), np.zeros(len(X_test_base))
    oof_xgb, test_xgb = np.zeros(len(X)), np.zeros(len(X_test_base))
    oof_cb, test_cb = np.zeros(len(X)), np.zeros(len(X_test_base))
    oof_rf, test_rf = np.zeros(len(X)), np.zeros(len(X_test_base))

    print(f"\n    {stage_name}: 10-fold base training" + (f" with {len(pseudo_X)} pseudo rows" if has_pseudo else ""))

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        print(f"      -> Processing Fold {fold+1}/{n_splits}")

        X_tr_raw = X.iloc[tr_idx].copy()
        X_va_raw = X.iloc[va_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        y_va = y.iloc[va_idx].copy()

        # Leakage guard: pseudo rows are appended only to training data.
        # X_va_raw/y_va stay original and remain the only early-stopping target.
        if has_pseudo:
            X_tr_raw = pd.concat([X_tr_raw, pseudo_X], axis=0)
            y_tr = pd.concat([y_tr, pseudo_y], axis=0)

        X_tr, X_va, X_te = process_fold(X_tr_raw, X_va_raw, X_test_base, y_tr)

        # 1. LightGBM
        m_lgb = LGBMClassifier(**lgb_params)
        m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[early_stopping(100, verbose=False), log_evaluation(0)])
        oof_lgb[va_idx] = m_lgb.predict_proba(X_va)[:, 1]
        test_lgb += m_lgb.predict_proba(X_te)[:, 1] / n_splits

        # 2. XGBoost
        m_xgb = XGBClassifier(**xgb_params)
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof_xgb[va_idx] = m_xgb.predict_proba(X_va)[:, 1]
        test_xgb += m_xgb.predict_proba(X_te)[:, 1] / n_splits

        # 3. CatBoost
        cat_idx = [X_tr.columns.get_loc(c) for c in low_card_cats]
        train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
        val_pool = Pool(X_va, y_va, cat_features=cat_idx)
        test_pool = Pool(X_te, cat_features=cat_idx)
        m_cb = CatBoostClassifier(**cb_params)
        m_cb.fit(train_pool, eval_set=val_pool)
        oof_cb[va_idx] = m_cb.predict_proba(val_pool)[:, 1]
        test_cb += m_cb.predict_proba(test_pool)[:, 1] / n_splits

        # 4. Random Forest (Orthogonal Variance)
        X_tr_rf, X_va_rf, X_te_rf = X_tr.copy(), X_va.copy(), X_te.copy()
        for col in low_card_cats:
            X_tr_rf[col] = X_tr_rf[col].cat.codes
            X_va_rf[col] = X_va_rf[col].cat.codes
            X_te_rf[col] = X_te_rf[col].cat.codes

        # Scikit-learn NaN Trap Fix
        X_tr_rf = X_tr_rf.fillna(-999)
        X_va_rf = X_va_rf.fillna(-999)
        X_te_rf = X_te_rf.fillna(-999)

        m_rf = RandomForestClassifier(n_estimators=1000, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1)
        m_rf.fit(X_tr_rf, y_tr)
        oof_rf[va_idx] = m_rf.predict_proba(X_va_rf)[:, 1]
        test_rf += m_rf.predict_proba(X_te_rf)[:, 1] / n_splits

    return {
        'LGBM': oof_lgb,
        'XGB': oof_xgb,
        'CatBoostFoldTE': oof_cb,
        'RF': oof_rf
    }, {
        'LGBM': test_lgb,
        'XGB': test_xgb,
        'CatBoostFoldTE': test_cb,
        'RF': test_rf
    }


def run_ultimate_catboost_cv(pseudo_test_indices=None, pseudo_y=None, stage_name='Clean ultimate CatBoost'):
    X_ult, X_test_ult, y_ult, cat_cols = build_ultimate_catboost_matrix()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    has_pseudo = pseudo_test_indices is not None and pseudo_y is not None and len(pseudo_test_indices) > 0

    oof = np.zeros(len(X_ult))
    test_pred = np.zeros(len(X_test_ult))
    pseudo_X_ult = None

    if has_pseudo:
        pseudo_X_ult = X_test_ult.iloc[pseudo_test_indices].copy()
        pseudo_X_ult.index = pd.RangeIndex(start=len(X_ult), stop=len(X_ult) + len(pseudo_X_ult))

    print(f"\n    {stage_name}: 5-fold historical CatBoost" + (f" with {len(pseudo_test_indices)} pseudo rows" if has_pseudo else ""))

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_ult, y_ult)):
        print(f"      -> Ultimate CatBoost Fold {fold+1}/{cv.n_splits}")
        X_tr = X_ult.iloc[tr_idx].copy()
        y_tr = y_ult.iloc[tr_idx].copy()
        X_va = X_ult.iloc[va_idx].copy()
        y_va = y_ult.iloc[va_idx].copy()

        # Leakage guard: pseudo rows never enter the validation fold or early-stopping target.
        if has_pseudo:
            X_tr = pd.concat([X_tr, pseudo_X_ult], axis=0)
            y_tr = pd.concat([y_tr, pseudo_y], axis=0)

        model = CatBoostClassifier(**ultimate_cb_params, cat_features=cat_cols)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50)
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_pred += model.predict_proba(X_test_ult)[:, 1] / cv.n_splits

    return oof, test_pred


def fit_lr_meta(oof_dict, test_dict, stage_name, class_weight='balanced'):
    names = list(oof_dict.keys())
    meta_X = np.column_stack([oof_dict[name] for name in names])
    meta_test_X = np.column_stack([test_dict[name] for name in names])

    assert meta_X.shape == (len(X), len(names))
    assert meta_test_X.shape == (len(X_test_base), len(names))

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=2026)
    meta_oof = np.zeros(len(y))

    for tr_idx, va_idx in cv.split(meta_X, y):
        fold_meta = LogisticRegressionCV(
            cv=10,
            scoring='roc_auc',
            class_weight=class_weight,
            max_iter=5000,
            n_jobs=-1,
            random_state=42
        )
        fold_meta.fit(meta_X[tr_idx], y.iloc[tr_idx])
        meta_oof[va_idx] = fold_meta.predict_proba(meta_X[va_idx])[:, 1]

    final_meta = LogisticRegressionCV(
        cv=10,
        scoring='roc_auc',
        class_weight=class_weight,
        max_iter=5000,
        n_jobs=-1,
        random_state=42
    )
    final_meta.fit(meta_X, y)
    meta_test_preds = final_meta.predict_proba(meta_test_X)[:, 1]

    label = 'balanced' if class_weight == 'balanced' else 'plain'
    print(f"      {stage_name} LogisticRegressionCV ({label}) ROC-AUC: {roc_auc_score(y, meta_oof):.5f}")
    print("      Coefficients: " + " | ".join([f"{name}={coef:.4f}" for name, coef in zip(names, final_meta.coef_[0])]))

    return meta_oof, meta_test_preds


def rank01(values):
    return pd.Series(values).rank(method='average').to_numpy() / len(values)


def search_rank_blend(oof_dict, test_dict, stage_name, n_iter=25000, seed=42):
    names = list(oof_dict.keys())
    rank_oof = np.column_stack([rank01(oof_dict[name]) for name in names])
    rank_test = np.column_stack([rank01(test_dict[name]) for name in names])

    candidate_weights = []
    candidate_weights.extend(np.eye(len(names)))
    candidate_weights.append(np.ones(len(names)) / len(names))

    rng = np.random.default_rng(seed)
    for _ in range(n_iter):
        candidate_weights.append(rng.dirichlet(np.ones(len(names))))

    best_auc = -np.inf
    best_w = None
    for w in candidate_weights:
        pred = rank_oof @ w
        auc = roc_auc_score(y, pred)
        if auc > best_auc:
            best_auc = auc
            best_w = w

    blend_oof = rank_oof @ best_w
    blend_test = rank_test @ best_w
    print(f"      {stage_name} no-SciPy rank blend ROC-AUC: {best_auc:.5f}")
    print("      Rank weights: " + " | ".join([f"{name}={weight:.4f}" for name, weight in zip(names, best_w)]))
    return blend_oof, blend_test


def collect_candidates(oof_dict, test_dict, stage_name):
    candidates = {}

    print(f"\n    {stage_name} base OOF ROC-AUCs:")
    for name in oof_dict:
        auc = roc_auc_score(y, oof_dict[name])
        print(f"      {name}: {auc:.5f}")
        candidates[f'{stage_name}_{name}'] = (auc, oof_dict[name], test_dict[name])

    meta_bal_oof, meta_bal_test = fit_lr_meta(oof_dict, test_dict, stage_name, class_weight='balanced')
    candidates[f'{stage_name}_LR_balanced'] = (roc_auc_score(y, meta_bal_oof), meta_bal_oof, meta_bal_test)

    meta_plain_oof, meta_plain_test = fit_lr_meta(oof_dict, test_dict, stage_name, class_weight=None)
    candidates[f'{stage_name}_LR_plain'] = (roc_auc_score(y, meta_plain_oof), meta_plain_oof, meta_plain_test)

    rank_oof, rank_test = search_rank_blend(oof_dict, test_dict, stage_name)
    candidates[f'{stage_name}_RankBlend'] = (roc_auc_score(y, rank_oof), rank_oof, rank_test)

    best_name, (best_auc, _, _) = max(candidates.items(), key=lambda kv: kv[1][0])
    print(f"      -> Best {stage_name} candidate: {best_name} ({best_auc:.5f})")
    return candidates


def build_stratified_pseudo_labels(test_probs, pos_threshold=0.90, neg_threshold=0.10):
    pos_candidates = np.flatnonzero(test_probs >= pos_threshold)
    neg_candidates = np.flatnonzero(test_probs <= neg_threshold)

    available_pos = len(pos_candidates)
    available_neg = len(neg_candidates)
    pos_rate = float(y.mean())
    neg_rate = 1.0 - pos_rate

    if available_pos == 0 or available_neg == 0:
        capped_pos, capped_neg = 0, 0
    else:
        max_total = int(min(available_pos / pos_rate, available_neg / neg_rate))
        capped_pos = min(available_pos, int(round(max_total * pos_rate)))
        capped_neg = min(available_neg, max_total - capped_pos)

        # Preserve at least one row for each class when both candidate pools exist.
        if max_total >= 2:
            capped_pos = max(1, capped_pos)
            capped_neg = max(1, capped_neg)

        while capped_pos > available_pos:
            capped_pos -= 1
        while capped_neg > available_neg:
            capped_neg -= 1

    print("\n[4/6] Stratified pseudo-label bottleneck check...")
    print(f"    Available class-1 candidates >= {pos_threshold:.2f}: {available_pos}")
    print(f"    Available class-0 candidates <= {neg_threshold:.2f}: {available_neg}")
    print(f"    Empirical target ratio: {pos_rate:.2%} class-1 / {neg_rate:.2%} class-0")
    print(f"    Selected after bottleneck: {capped_pos} positives, {capped_neg} negatives")

    if capped_pos == 0 or capped_neg == 0:
        empty_X = X_test_base.iloc[[]].copy()
        empty_y = pd.Series([], dtype=y.dtype, name=y.name)
        return empty_X, empty_y, np.array([], dtype=int)

    selected_pos_idx = pos_candidates[np.argsort(-test_probs[pos_candidates])[:capped_pos]]
    selected_neg_idx = neg_candidates[np.argsort(test_probs[neg_candidates])[:capped_neg]]
    selected_idx = np.concatenate([selected_pos_idx, selected_neg_idx])
    selected_labels = np.concatenate([
        np.ones(capped_pos, dtype=int),
        np.zeros(capped_neg, dtype=int)
    ])

    pseudo_X = X_test_base.iloc[selected_idx].copy()
    pseudo_X.index = pd.RangeIndex(start=len(X), stop=len(X) + len(pseudo_X))
    pseudo_y = pd.Series(selected_labels, index=pseudo_X.index, name=y.name).astype(y.dtype)

    print(f"    Pseudo-label positive rate after bottleneck: {pseudo_y.mean():.2%}")
    return pseudo_X, pseudo_y, selected_idx


print("\n[2/6] Executing clean 10-Fold CV with Orthogonal Random Forest...")
clean_oof_dict, clean_test_dict = run_base_cv(stage_name='Clean')

print("\n[3/6] Executing historical high-OOF CatBoost branch...")
ultimate_oof, ultimate_test = run_ultimate_catboost_cv(stage_name='Clean ultimate CatBoost')
clean_oof_dict['UltimateCatBoost'] = ultimate_oof
clean_test_dict['UltimateCatBoost'] = ultimate_test

print("\n[3/6] Training/evaluating clean in-memory stacks and rank blends...")
clean_candidates = collect_candidates(clean_oof_dict, clean_test_dict, stage_name='Clean')

raw_selector_names = [name for name in clean_candidates if not name.endswith('RankBlend')]
pseudo_selector_name = max(raw_selector_names, key=lambda name: clean_candidates[name][0])
pseudo_selector_auc, _, pseudo_selector_test = clean_candidates[pseudo_selector_name]
print(f"\n    Pseudo selector: {pseudo_selector_name} ({pseudo_selector_auc:.5f})")
pseudo_X, pseudo_y, pseudo_test_indices = build_stratified_pseudo_labels(pseudo_selector_test)

all_candidates = dict(clean_candidates)

if len(pseudo_X) > 0:
    print("\n[5/6] Re-running models with pseudo labels appended only to training folds...")
    pseudo_oof_dict, pseudo_test_dict = run_base_cv(
        pseudo_X=pseudo_X,
        pseudo_y=pseudo_y,
        stage_name='Pseudo-augmented'
    )

    pseudo_ultimate_oof, pseudo_ultimate_test = run_ultimate_catboost_cv(
        pseudo_test_indices=pseudo_test_indices,
        pseudo_y=pseudo_y,
        stage_name='Pseudo-augmented ultimate CatBoost'
    )
    pseudo_oof_dict['UltimateCatBoost'] = pseudo_ultimate_oof
    pseudo_test_dict['UltimateCatBoost'] = pseudo_ultimate_test

    print("\n[5/6] Training/evaluating pseudo-augmented stacks and rank blends...")
    pseudo_candidates = collect_candidates(pseudo_oof_dict, pseudo_test_dict, stage_name='Pseudo')
    all_candidates.update(pseudo_candidates)
else:
    print("\n[5/6] No pseudo rows survived the empirical bottleneck; retaining clean candidates only.")

best_name, (best_auc, final_oof, final_test_preds) = max(all_candidates.items(), key=lambda kv: kv[1][0])
print(f"\n[!] SELECTED FINAL CANDIDATE: {best_name}")
print(f"[!] FINAL SELECTED OOF ROC-AUC: {best_auc:.5f}")

print("\n[6/6] Saving final tournament predictions...")
submission = test[['Id']].copy()
submission['Drafted'] = final_test_preds
submission.to_csv('Data/submission_1st_Place.csv', index=False)
print("SUCCESS: Saved to 'Data/submission_1st_Place.csv'")


--- FINAL PHASE: HIGH-CEILING STACK + CATBOOST TOURNAMENT + LEAK-SAFE PSEUDO ---
Local target split: 1803 class-1 / 978 class-0 (64.83% class-1).
Note: pseudo-label bottleneck uses this local empirical split, not the stale 20/80 assumption.

[1/6] Running 3-Way Split Optuna Tuning...

[2/6] Executing clean 10-Fold CV with Orthogonal Random Forest...

    Clean: 10-fold base training
      -> Processing Fold 1/10
      -> Processing Fold 2/10
      -> Processing Fold 3/10
      -> Processing Fold 4/10
      -> Processing Fold 5/10
      -> Processing Fold 6/10
      -> Processing Fold 7/10
      -> Processing Fold 8/10
      -> Processing Fold 9/10
      -> Processing Fold 10/10

[3/6] Executing historical high-OOF CatBoost branch...

    Clean ultimate CatBoost: 5-fold historical CatBoost
      -> Ultimate CatBoost Fold 1/5
      -> Ultimate CatBoost Fold 2/5
      -> Ultimate CatBoost Fold 3/5
      -> Ultimate CatBoost Fold 4/5
      -> Ultimate CatBoost Fold 5/5

[3/6] Training/eval

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

print("--- LEGAL FAST SUBMISSION: HISTORICAL BEST CATBOOST MATRIX ---")

train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')
y = train['Drafted'].reset_index(drop=True)
cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']

def build_legal_catboost_matrix(train, test):
    df = pd.concat([train.drop(columns=['Drafted']), test], axis=0).reset_index(drop=True)
    physical_metrics = ['Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
                        'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'Age']

    for col in physical_metrics:
        if df[col].isnull().sum() > 0:
            df[f'{col}_Is_Missing'] = df[col].isnull().astype(int)

    for col in physical_metrics:
        df[col] = df[col].fillna(df.groupby('Position')[col].transform('median'))
        df[col] = df[col].fillna(df[col].median())

    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Speed_Weight_Ratio'] = df['Weight'] / df['Sprint_40yd']
    df['Explosiveness_Index'] = df['Vertical_Jump'] * df['Broad_Jump']
    df['Power_Score'] = df['Bench_Press_Reps'] * df['Weight']
    df['School_Combine_Count'] = df.groupby('School')['Id'].transform('count')
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)

    for col in physical_metrics:
        df[f'{col}_Pos_Mean'] = df.groupby('Position')[col].transform('mean')
        df[f'{col}_Pos_Diff'] = df[col] - df[f'{col}_Pos_Mean']
        df.drop(columns=[f'{col}_Pos_Mean'], inplace=True)

    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype('category')

    X_train = df.iloc[:len(train)].drop(columns=['Id']).reset_index(drop=True)
    X_test = df.iloc[len(train):].drop(columns=['Id']).reset_index(drop=True)
    return X_train, X_test

X_train, X_test = build_legal_catboost_matrix(train, test)

best_catboost_params = {
    'iterations': 1183,
    'learning_rate': 0.010492928203031996,
    'depth': 4,
    'l2_leaf_reg': 4.5594175959311425,
    'min_data_in_leaf': 7,
    'random_seed': 42,
    'verbose': 0,
    'allow_writing_files': False
}

model = CatBoostClassifier(**best_catboost_params, cat_features=cat_cols)
model.fit(X_train, y)
preds = model.predict_proba(X_test)[:, 1]

submission = test[['Id']].copy()
submission['Drafted'] = preds
submission.to_csv('Data/submission_legal_catboost_full.csv', index=False)
submission.to_csv('Data/submission_1st_Place.csv', index=False)

print("SUCCESS: Saved legal submissions:")
print("  Data/submission_legal_catboost_full.csv")
print("  Data/submission_1st_Place.csv")
print(submission['Drafted'].describe())
